# 🧠 04 · gradient로 SNN 학습하기 — surrogate·BPTT에서 지역 학습(e-prop·FA·DECOLLE)으로

03편의 R-STDP에서 세 번째 인자는 **스칼라 보상 하나**였습니다. 그런데 깊거나 뉴런이 많은 망에서는 "각 뉴런이 최종 오차에 *얼마나* 기여했나"라는 정교한 credit을 전역 보상 하나로 나눠 주기 어렵습니다. 이번 편은 그 credit을 **과제 오차의 gradient**로 계산하는 계열, 즉 backprop 쪽 방법들을 다룹니다.

걸림돌은 둘입니다. ① 스파이크는 미분이 안 됩니다(§1에서 **surrogate gradient**로 우회). ② 정확한 gradient를 주는 **BPTT**는 생물학적으로 불편한 세 가지를 안고 있습니다(§3: ⓐ 오프라인 ⓑ 전체 히스토리 저장 ⓒ weight transport). 그래서 이 편의 줄기는 하나입니다 — **같은 gradient를, 점점 더 지역적이고 생물학적으로 그럴듯하게 만들기.** 그 사다리를 한 칸씩 오릅니다.

| 절 | 방법 | BPTT의 무엇을 없애나 |
|---|---|---|
| §1·§3 | surrogate + **BPTT** | (기준점: 정확하지만 ⓐⓑⓒ를 다 안고 있음) |
| §4–5 | **e-prop** | ⓐ 오프라인 · ⓑ 전체 저장 — 시간 방향을 순방향 자격 흔적으로 |
| §6 | **feedback alignment** | ⓒ weight transport — 되쏘는 가중치를 고정 랜덤 $B$로 |
| §7 | **DECOLLE** | 오차 전파 자체 — 층마다 국소 손실로(공간적으로도 지역) |

이 편의 중심은 **e-prop**(Bellec et al., 2020)입니다. **e-prop은 BPTT의 지역·온라인 근사**로, 03편의 자격 흔적은 그대로 두고 세 번째 인자만 **과제 오차에서 유도한 뉴런별 학습 신호** $L_j^t$로 바꿉니다. 그러면 전체 gradient가 이렇게 분해됩니다.

$$ \frac{dE}{dW_{ji}} = \sum_t \underbrace{L_j^t}_{\text{③ 학습 신호}} \cdot \underbrace{e_{ji}^t}_{\text{①×② 자격 흔적}} $$

- $E$: 과제 손실(오차), $W_{ji}$: 뉴런 $j$의 입력 시냅스 $i$의 가중치, $t$: 시간 스텝.
- $e_{ji}^t$: **순방향으로** 계산되는 지역 자격 흔적(과거를 저장하지 않음).
- $L_j^t$: 출력 오차를 readout으로 되쏘아 만든 신호(시간 역방향 패스 아님).

**자격 흔적(지역) × 학습 신호(오차)** 라는 이 골격은 03편의 *자격 흔적 × 보상*과 한 줄기이고, §6·§7은 그 "학습 신호"를 어떻게 더 지역적으로 만들지에 대한 이야기입니다.

## 이번 편 학습 목표
1. **surrogate gradient**로 스파이크의 미분 불가를 우회하고(§1), **BPTT**가 무엇을 하며 왜 비지역·오프라인인지 이해한다(§3).
2. **e-prop**을 손으로 구현해 그 gradient가 BPTT와 **정렬**됨을 확인하고, e-prop만으로 SNN을 학습한다(§4–5).
3. **feedback alignment**로 weight transport를, **DECOLLE**로 층 사이 오차 전파를 없애 — gradient 학습을 점점 더 지역적으로 만든다(§6–7).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (9, 3.2)
plt.rcParams["axes.grid"] = True
device = torch.device("cpu")

# --- 네트워크/뉴런 설정 ---
n_in, H, C = 30, 50, 3        # 입력 / 은닉 LIF / 클래스
T = 40                        # 시간 스텝
alpha, v_th, gamma = 0.9, 1.0, 0.3   # 막전위 감쇠, 임계값, surrogate 크기
print("SNN:", n_in, "->", H, "(LIF) ->", C, "| T =", T)

## 1. 스파이크의 미분 불가 문제와 surrogate gradient

스파이크 $z = H(v - v_{th})$($H$는 계단함수, $v$는 막전위, $v_{th}$는 임계값)는 미분이 0 또는 ∞라 gradient가 흐르지 않습니다. 해결책은 **역방향에서만** 매끈한 가짜 미분(pseudo-derivative/surrogate)을 쓰는 것:

$$ \psi_j^t = \gamma \cdot \max\!\Big(0,\; 1 - \frac{|v_j^t - v_{th}|}{v_{th}}\Big) \quad(\text{삼각형 surrogate}) $$

여기서 $\psi_j^t$는 뉴런 $j$의 시각 $t$에서의 가짜 미분, $\gamma$는 그 크기(스케일)입니다. 이 $\psi$는 **BPTT와 e-prop 둘 다에서** 쓰입니다 — e-prop은 이 $\psi$를 아래 eligibility trace 계산에 사용합니다.

아래는 forward는 계단함수, backward는 $\psi$를 흘리는 커스텀 autograd 함수입니다.

In [ ]:
class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v):
        ctx.save_for_backward(v)
        return (v >= v_th).float()          # 순전파: 계단함수
    @staticmethod
    def backward(ctx, g):
        (v,) = ctx.saved_tensors
        psi = gamma*torch.clamp(1 - torch.abs(v - v_th)/v_th, min=0)  # 역전파: 삼각 surrogate
        return g*psi
spike = SpikeFn.apply

# surrogate 모양 시각화
vv = torch.linspace(-0.5, 2.5, 300)
psi = gamma*torch.clamp(1 - torch.abs(vv - v_th)/v_th, min=0)
fig, ax = plt.subplots(figsize=(7,2.8))
ax.plot(vv, (vv>=v_th).float(), color="tab:gray", label="forward: 계단함수 H(v-θ)")
ax.plot(vv, psi, color="tab:red", lw=2, label=r"backward: surrogate $\psi$")
ax.axvline(v_th, ls=":", c="k"); ax.set_xlabel("막전위 v"); ax.set_title("스파이크: 순전파는 계단, 역전파는 surrogate")
ax.legend(); plt.tight_layout(); plt.show()

## 2. 과제와 forward pass

간단한 **3-클래스 스파이크 패턴 분류**: 클래스 $c$면 입력의 $c$번째 1/3 그룹이 포아송 발화합니다. readout은 은닉 스파이크를 시간에 걸쳐 적분해 로짓을 만듭니다.

forward 루프에서 e-prop에 필요한 세 가지를 함께 기록합니다:
- **eligibility vector** $\varepsilon_i^t = \alpha\,\varepsilon_i^{t-1} + x_i^t$ — presynaptic 스파이크 $x_i^t$의 저역통과 흔적 ($\alpha$는 막전위 감쇠 계수). 입력 시냅스에선 이 값이 뒤쪽 뉴런 $j$와 무관합니다.
- **pseudo-derivative** $\psi_j^t$.
- 은닉 스파이크 $z_j^t$ (readout 학습용).

In [ ]:
def gen_batch(B):
    y = torch.randint(0, C, (B,))
    x = torch.zeros(T, B, n_in)
    per = n_in // C
    for b in range(B):
        c = y[b].item()
        x[:, b, c*per:(c+1)*per] = (torch.rand(T, per) < 0.5).float()
    return x, y

def forward_full(x, W_in, W_out):
    # autograd 가능한 forward. logits + e-prop 중간값(psi, eps, z) 반환.
    B = x.shape[1]
    v = torch.zeros(B, H); z = torch.zeros(B, H); out = torch.zeros(B, C)
    eps = torch.zeros(B, n_in)
    psi_rec, eps_rec, z_rec = [], [], []
    for t in range(T):
        I = x[t] @ W_in.t()
        v = alpha*v + I - z*v_th        # LIF (soft reset)
        z = spike(v)
        out = out + z @ W_out.t()        # readout: 은닉 스파이크 적분
        with torch.no_grad():            # e-prop 중간값 (autograd와 분리 기록)
            psi = gamma*torch.clamp(1 - torch.abs(v - v_th)/v_th, min=0)
            eps = alpha*eps + x[t]
            psi_rec.append(psi.clone()); eps_rec.append(eps.clone()); z_rec.append(z.detach().clone())
    return out / T, psi_rec, eps_rec, z_rec

x, y = gen_batch(4)
logits, *_ = forward_full(x, torch.rand(H,n_in)*0.3, torch.rand(C,H)*0.3-0.15)
print("logits shape:", tuple(logits.shape), "(batch x class)")

## 3. BPTT — e-prop이 근사하는 기준

BPTT(Backpropagation Through Time)는 위 forward 그래프를 **시간축으로 다 펼친 뒤 끝에서부터 gradient를 역방향으로** 흘립니다. PyTorch autograd로는 `loss.backward()` 한 줄입니다.

하지만 이게 뉴로모픽 관점에서 불편한 이유:
- **오프라인**: 시퀀스가 끝나야 backward 시작 (온라인 아님).
- **전체 히스토리 저장**: 모든 시점의 활성값을 메모리에 들고 있어야 함.
- **비지역(weight transport)**: gradient가 순전파와 같은 가중치를 거꾸로 타고 흘러야 함 — 생물학적으로 구현 곤란.

e-prop은 **같은 gradient를 순방향·지역적으로** 만들어 이 중 ①·②를 없앱니다. (③ weight transport는 학습 신호가 readout 가중치를 되쓰는 형태로 일부 남는데, §6 feedback alignment에서 없앱니다.) 먼저 BPTT gradient를 기준값으로 구해둡니다.

In [ ]:
W_in0  = torch.rand(H, n_in)*0.3
W_out0 = torch.rand(C, H)*0.3 - 0.15
x, y = gen_batch(128)

Wi = W_in0.clone().requires_grad_(True)
Wo = W_out0.clone().requires_grad_(True)
logits, *_ = forward_full(x, Wi, Wo)
loss = torch.nn.functional.cross_entropy(logits, y)
loss.backward()                              # ★ BPTT (autograd 한 줄)
gWin_bptt, gWout_bptt = Wi.grad.clone(), Wo.grad.clone()
print("BPTT gradient 계산 완료. |gW_in| =", round(gWin_bptt.norm().item(), 4))

## 4. e-prop — 순방향·지역으로 같은 gradient 만들기

이제 backward 없이, forward에서 기록한 것들만으로 gradient를 조립합니다:

1. **학습 신호** $L_j = \sum_k W^{out}_{kj}\,(\pi_k - y^*_k)$ — 출력 오차를 readout 가중치로 되쏨(순간적 사영, 시간 역방향 아님). 여기서 $\pi_k$는 softmax 출력 확률, $y^*_k$는 정답 원-핫, $W^{out}$은 readout(출력층) 가중치입니다.
2. **은닉 자격 흔적** $\sum_t e_{ji}^t = \sum_t \psi_j^t\,\varepsilon_i^t$ (앞서의 pseudo-derivative $\psi$ × eligibility vector $\varepsilon$).
3. **gradient** $\;\dfrac{dE}{dW_{ji}} = \dfrac1T\sum_t L_j\,e_{ji}^t$.

그리고 BPTT gradient와 **cosine similarity**로 비교합니다.

In [ ]:
def eprop_grads(x, y, W_in, W_out):
    B = x.shape[1]
    logits, psi_rec, eps_rec, z_rec = forward_full(x, W_in.detach(), W_out.detach())
    pi = torch.softmax(logits, dim=1)
    onehot = torch.zeros(B, C); onehot[torch.arange(B), y] = 1.0
    err = (pi - onehot) / B                       # 출력 오차 [B,C]
    L = err @ W_out.detach()                       # ③ 학습 신호 [B,H]
    E = torch.zeros(B, H, n_in)                     # sum_t psi*eps
    for t in range(T):
        E += psi_rec[t].unsqueeze(2) * eps_rec[t].unsqueeze(1)
    gW_in  = (L.unsqueeze(2) * E).sum(0) / T        # [H,n_in]
    gW_out = err.t() @ torch.stack(z_rec).mean(0)   # [C,H]
    return gW_in, gW_out

gWin_ep, gWout_ep = eprop_grads(x, y, W_in0, W_out0)

def cos(a, b):
    a=a.flatten(); b=b.flatten(); return float((a@b)/(a.norm()*b.norm()+1e-9))
c_in, c_out = cos(gWin_ep, gWin_bptt), cos(gWout_ep, gWout_bptt)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(gWin_bptt.flatten(), gWin_ep.flatten(), s=6, alpha=0.4, color="tab:blue")
lim = max(gWin_bptt.abs().max(), gWin_ep.abs().max()).item()
ax[0].plot([-lim,lim],[-lim,lim], "r--", lw=1)
ax[0].set_xlabel("BPTT gradient (W_in)"); ax[0].set_ylabel("e-prop gradient (W_in)")
ax[0].set_title(f"W_in gradient: e-prop vs BPTT\ncosine = {c_in:.3f}")
ax[1].scatter(gWout_bptt.flatten(), gWout_ep.flatten(), s=12, alpha=0.6, color="tab:green")
lim2 = max(gWout_bptt.abs().max(), gWout_ep.abs().max()).item()
ax[1].plot([-lim2,lim2],[-lim2,lim2], "r--", lw=1)
ax[1].set_xlabel("BPTT gradient (W_out)"); ax[1].set_ylabel("e-prop gradient (W_out)")
ax[1].set_title(f"W_out gradient\ncosine = {c_out:.3f}")
plt.tight_layout(); plt.show()
print(f"cosine similarity — W_in: {c_in:.3f}  W_out: {c_out:.3f}")
print("=> 시간 역방향 없이, 지역·순방향으로 계산했는데 BPTT와 거의 같은 방향!")

> **정직한 주석**: 여기 cosine이 ~1.0인 건 이 네트워크가 **얕은 feedforward**(은닉 1층)라 e-prop이 생략하는 항(리셋·순환 Jacobian)이 작기 때문입니다. **순환 연결**(hidden→hidden)이나 **긴 시간의존성**이 생기면 e-prop은 진짜 *근사*가 되어 정렬도가 내려갑니다 — 그래도 학습은 잘 됩니다. (아래 '실험'에서 순환 항을 넣어 정렬도가 떨어지는 걸 볼 수 있습니다.)

## 5. e-prop만으로 학습하기

BPTT/autograd 없이, 위 e-prop gradient로 경사하강해 분류기를 학습합니다.

In [ ]:
def evaluate(W_in, W_out, B=400):
    xv, yv = gen_batch(B)
    logits, *_ = forward_full(xv, W_in, W_out)
    return (logits.argmax(1) == yv).float().mean().item()

W_in  = torch.rand(H, n_in)*0.3
W_out = torch.rand(C, H)*0.3 - 0.15
lr = 0.5
acc_hist = [(0, evaluate(W_in, W_out))]
for step in range(1, 301):
    xb, yb = gen_batch(64)
    gWin, gWout = eprop_grads(xb, yb, W_in, W_out)
    W_in  -= lr*gWin
    W_out -= lr*gWout
    if step % 20 == 0:
        acc_hist.append((step, evaluate(W_in, W_out)))

xs = [a[0] for a in acc_hist]; ys = [a[1] for a in acc_hist]
fig, ax = plt.subplots(figsize=(7.5,3.2))
ax.plot(xs, ys, marker="o", ms=3, color="tab:blue")
ax.axhline(1/3, ls="--", c="gray", label="우연 (0.33)")
ax.set_ylim(0,1.05); ax.set_xlabel("e-prop 업데이트 스텝"); ax.set_ylabel("정확도")
ax.set_title("e-prop 학습 곡선 (BPTT 없이)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"최종 정확도: {ys[-1]:.2f}")

## 6. feedback alignment — weight transport까지 없애기

§3에서 BPTT의 생물학적 문제 셋을 꼽았습니다: ① 오프라인 ② 전체 히스토리 저장 ③ **weight transport**(순전파에 쓴 가중치를 역방향에 그대로 재사용). e-prop은 자격 흔적을 순방향·지역으로 만들어 **①·②를 없앴습니다.** 그런데 ③은 정말 사라졌을까요?

§4의 학습 신호를 다시 봅니다.

$$ L_j = \sum_k W^{out}_{kj}\,(\pi_k - y^*_k) $$

기호는 §4에서 이어집니다.
- $j$: 은닉 뉴런 인덱스, $k$: 출력 클래스 인덱스. $\sum_k$ 는 모든 출력 클래스에 대해 더한다는 뜻입니다.
- $L_j$: 은닉 뉴런 $j$ 가 받는 **학습 신호** — "이 뉴런의 가중치를 어느 쪽으로 밀지"를 알려주는 한 숫자.
- $\pi_k$: 출력층 softmax 확률(모델이 클래스 $k$ 라고 믿는 정도), $y^*_k$: 정답 원-핫(맞는 클래스만 1, 나머지 0). 따라서 $\pi_k - y^*_k$ 가 곧 **출력 오차**(예측 − 정답)입니다.
- $W^{out}_{kj}$: 출력 클래스 $k$ 와 은닉 뉴런 $j$ 를 잇는 **readout(출력층) 가중치** — §2 순전파에서 은닉 스파이크를 클래스 점수로 바꾸던 바로 그 가중치입니다.

즉 이 식은 **출력 오차 $(\pi_k - y^*_k)$ 를 readout 가중치 $W^{out}$ 에 실어 은닉층으로 되쏘는** 것입니다. 규모는 작아도 이건 여전히 weight transport입니다 — 순전파에 썼던 바로 그 $W^{out}$ 를 역방향에서도 정확히 알아야 하니까요. 뇌에서 시냅스가 자기 순전파 파트너의 값을 거꾸로 아는 건 구현하기 어렵습니다.

**feedback alignment**(Lillicrap et al., 2016)의 답은 대담합니다: 그 되쏘는 가중치를 **고정된 랜덤 행렬 $B$** 로 바꿔 버립니다. $B$ 는 $W^{out}$ 과 아무 관계 없이 처음에 무작위로 정해 두고, 학습 내내 **한 번도 갱신하지 않습니다.**

$$ L_j^{\text{FA}} = \sum_k B_{kj}\,(\pi_k - y^*_k) \qquad (B:\ \text{고정 랜덤, 갱신 안 함}) $$

- $B_{kj}$: 출력 클래스 $k$ 에서 은닉 뉴런 $j$ 로 오차를 되쏘는 **고정 랜덤 피드백 가중치**. 위 식에서 $W^{out}_{kj}$ 가 있던 자리를 그대로 대신합니다.

놀랍게도 이걸로도 학습이 됩니다. 원문은 그 발견을 이렇게 적습니다.

> "a network can learn to extract useful information from signals sent through these random feedback connections." — Lillicrap et al. (2016)

왜 될까요? 학습이 진행되며 **순전파 가중치 $W^{out}$ 이 랜덤 $B$ 쪽으로 스스로 정렬(align)** 되기 때문입니다 — 그래서 $B$ 를 통과한 오차가 결국 유용한 방향을 가리키게 됩니다. 네트워크가 주어진 랜덤 피드백에 맞춰 "배우는 법을 배우는" 셈이고, 이 정렬이 이름 'feedback alignment'의 유래입니다.

아래에서 §5와 **똑같은 과제**를, 학습 신호만 $L^{\text{FA}} = \text{err}\cdot B$ 로 바꿔 돌립니다. 두 가지를 봅니다: ① 랜덤 피드백만으로 정확도가 오르는지, ② $W^{out}$ 과 $B$ 의 정렬(코사인)이 0 근처에서 올라가는지.

> 출력 오차를 각 은닉층으로 **랜덤 가중치로 한 번에 방송**하는 변형이 **DFA**(Direct Feedback Alignment; Nøkland, 2016)입니다 — 깊은 망에서 특히 편리합니다.

In [ ]:
# feedback alignment: e-prop의 학습 신호에서 readout 가중치를 '고정 랜덤 B'로 교체
def eprop_grads_fb(x, y, W_in, W_out, B_fb):
    Bn = x.shape[1]
    logits, psi_rec, eps_rec, z_rec = forward_full(x, W_in.detach(), W_out.detach())
    pi = torch.softmax(logits, 1)
    oh = torch.zeros(Bn, C); oh[torch.arange(Bn), y] = 1.0
    err = (pi - oh) / Bn
    L = err @ B_fb                                   # ★ W_out 대신 고정 랜덤 B (weight transport 제거)
    E = torch.zeros(Bn, H, n_in)
    for t in range(T): E += psi_rec[t].unsqueeze(2) * eps_rec[t].unsqueeze(1)
    gW_in  = (L.unsqueeze(2) * E).sum(0) / T
    gW_out = err.t() @ torch.stack(z_rec).mean(0)    # readout W_out은 평소대로 학습(순전파용)
    return gW_in, gW_out

def cos(a, b):
    a = a.flatten(); b = b.flatten(); return float((a @ b) / (a.norm()*b.norm() + 1e-9))

torch.manual_seed(1)
W_in  = torch.rand(H, n_in)*0.3
W_out = torch.rand(C, H)*0.3 - 0.15
B_fb  = torch.rand(C, H)*0.3 - 0.15                  # 고정 랜덤 피드백 — 아래 루프에서 절대 갱신 안 함
lr = 0.5
steps, accs, aligns = [0], [evaluate(W_in, W_out)], [cos(W_out, B_fb)]
for s in range(1, 201):
    xb, yb = gen_batch(64)
    gWin, gWout = eprop_grads_fb(xb, yb, W_in, W_out, B_fb)
    W_in  -= lr*gWin
    W_out -= lr*gWout                                 # B_fb는 건드리지 않는다
    if s % 10 == 0:
        steps.append(s); accs.append(evaluate(W_in, W_out)); aligns.append(cos(W_out, B_fb))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(steps, accs, marker="o", ms=3, color="tab:blue")
ax[0].axhline(1/3, ls="--", c="gray", label="우연 (0.33)")
ax[0].set_ylim(0, 1.05); ax[0].set_xlabel("업데이트 스텝"); ax[0].set_ylabel("정확도")
ax[0].set_title("feedback alignment로 학습 (랜덤 피드백만으로)"); ax[0].legend()
ax[1].plot(steps, aligns, marker="o", ms=3, color="tab:red")
ax[1].set_xlabel("업데이트 스텝"); ax[1].set_ylabel(r"$\cos(W^{out},\,B)$")
ax[1].set_title("정렬: W_out이 랜덤 B로 회전해 감 (FA의 핵심)")
plt.tight_layout(); plt.show()
print(f"정확도: {accs[0]:.2f}(시작) -> {accs[-1]:.2f}(끝)")
print(f"정렬 cos(W_out,B): {aligns[0]:.3f}(랜덤, ~0) -> {aligns[-1]:.3f}   <- 이 상승이 FA가 되는 이유")

## 7. DECOLLE — 층마다 자기 손실로 배우기 (공간적으로도 지역)

앞의 세 방법은 모두 **하나의 전역 손실 $E$**(출력에서 잰 오차)에서 출발했습니다. BPTT는 그것을 시간·공간으로 다 펼쳐 역전파했고, e-prop은 그 시간 방향을 순방향 자격 흔적으로, feedback alignment는 공간 방향의 weight transport를 랜덤 $B$ 로 각각 풀었습니다. 그래도 공통점이 하나 남습니다: **오차는 언제나 출력층에서 시작해 은닉층으로 흘러들어왔다**는 것.

**DECOLLE**(Deep Continuous Local Learning; Kaiser, Mostafa & Neftci, 2020)는 그 마지막 고리를 끊습니다. **각 은닉층에 자기만의 손실을 붙여**, 층이 출력의 오차를 아예 보지 않고 **자기 국소 손실만으로** 배우게 합니다.

층 $l$ 의 (시간에 걸쳐 모은) 발화 활동을 $s^l$ 이라 하고, 그 층에 **고정된 랜덤 readout** $Y^l$ 을 하나 달아 국소 예측을 만듭니다.

$$ \hat{y}^{\,l} = Y^l\, s^l $$

- $l$: 층 번호. $s^l$: 층 $l$ 의 발화 활동(발화율 벡터).
- $Y^l$: 층 $l$ 전용 **고정 랜덤 readout 행렬**(학습하지 않음). feedback alignment의 랜덤 $B$ 와 같은 재료지만, 여기선 오차를 되쏘는 통로가 아니라 **그 층의 국소 목표를 만드는 도구**로 씁니다.
- $\hat{y}^{\,l}$: 층 $l$ 이 자기 힘으로 내놓는 국소 예측(클래스 점수).

이 국소 예측과 정답으로 **층마다 독립된 손실**을 정의하고,

$$ \mathcal{L}^l = \ell\big(\hat{y}^{\,l},\, y^*\big) $$

- $y^*$: 과제의 정답(원-핫). 모든 층이 같은 최종 정답을 각자의 국소 목표로 공유합니다.
- $\ell$: 손실 함수(여기선 교차 엔트로피). $\mathcal{L}^l$: 층 $l$ 의 국소 손실.

층 $l$ 의 가중치 $W^l$ 은 **오직 자기 손실 $\mathcal{L}^l$ 의 (surrogate) gradient** 로만 갱신합니다.

$$ \Delta W^l \;\propto\; -\,\frac{\partial \mathcal{L}^l}{\partial W^l} $$

$\mathcal{L}^l$ 은 층 $l$ 의 출력 $\hat{y}^{\,l}$ 과 정답 $y^*$ 에만 의존하므로, 이 미분에는 **다른 층의 값도, 뒤에서 흘러온 오차도 들어오지 않습니다.** 그래서 공간적으로 완전히 지역이고(층 사이 오차 전파 없음), 매 시점 순방향으로 계산돼 온라인입니다(BPTT처럼 시퀀스를 되감지 않음). 원문은 DECOLLE를 이렇게 요약합니다.

> "DECOLLE is capable of learning deep spatio-temporal representations from spikes relying solely on local information." — Kaiser et al. (2020)

이점은 분명합니다: 층 사이로 오차를 나르지 않으니 **깊은 망에서도 각 층이 서로 기다리지 않고 병렬로, 뉴로모픽 하드웨어에서 값싸게** 학습합니다. DECOLLE의 진가는 층이 여럿일 때 나오지만, 아래에선 ch04의 은닉 한 층으로 그 원리 — "출력 오차 없이 국소 손실만으로" — 를 확인합니다. 은닉층에 고정 랜덤 $Y$ 를 달고 국소 손실만으로 $W_{in}$ 을 학습시켜, 출력 오차를 한 번도 보지 않은 은닉 특징이 실제로 분류 가능해지는지 봅니다.

In [ ]:
# DECOLLE: 은닉층에 '고정 랜덤 국소 readout' Y를 달아, 국소 손실만으로 W_in 학습
# (출력층 W_out의 오차는 은닉층에 전혀 전달되지 않는다 — 공간적으로 지역)
def forward_hidden(x, W_in):                      # 은닉 발화율만 (W_in에 대해 grad 유지)
    B = x.shape[1]; v = torch.zeros(B, H); z = torch.zeros(B, H); rate = torch.zeros(B, H)
    for t in range(T):
        I = x[t] @ W_in.t(); v = alpha*v + I - z*v_th; z = spike(v); rate = rate + z
    return rate / T

torch.manual_seed(1)
W_in = (torch.rand(H, n_in)*0.3).requires_grad_(True)
Y = torch.rand(C, H)*0.6 - 0.3                     # 고정 랜덤 국소 readout Y^l (절대 학습 안 함)
lr = 0.5

def local_test_acc(B=400):
    xv, yv = gen_batch(B)
    with torch.no_grad():
        rate = forward_hidden(xv, W_in.detach())
        return ((rate @ Y.t()).argmax(1) == yv).float().mean().item()

steps, accs = [0], [local_test_acc()]
for s in range(1, 151):
    xb, yb = gen_batch(64)
    rate = forward_hidden(xb, W_in)                # 순방향 (surrogate 스파이크)
    logits_local = rate @ Y.t()                    # 국소 예측  y^l = Y^l s^l
    loss = torch.nn.functional.cross_entropy(logits_local, yb)   # 국소 손실 L^l = CE(y^l, y*)
    loss.backward()                                # ∂L^l/∂W_in — 이 층 안에서만
    with torch.no_grad():
        W_in -= lr * W_in.grad; W_in.grad.zero_()  # W_out은 관여하지 않음
    if s % 25 == 0:
        steps.append(s); accs.append(local_test_acc())

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(steps, accs, marker="o", ms=4, color="tab:green")
ax.axhline(1/3, ls="--", c="gray", label="우연 (0.33)")
ax.set_ylim(0, 1.05); ax.set_xlabel("업데이트 스텝"); ax.set_ylabel("국소 readout 정확도")
ax.set_title("DECOLLE: 은닉층이 출력 오차 없이 국소 손실만으로 학습"); ax.legend()
plt.tight_layout(); plt.show()
print(f"국소 readout 정확도: {accs[0]:.2f}(시작) -> {accs[-1]:.2f}(끝)")
print("=> 은닉층은 W_out의 오차를 한 번도 보지 않았다 — 자기 국소 손실만으로 분류 가능한 특징을 학습")

## 8. 실험 — e-prop은 언제 무너지나 (순환 연결)

§4의 '정직한 주석'에서, e-prop이 BPTT와 거의 일치한 건 이 망이 **얕은 피드포워드**(은닉 1층, 순환 없음)라 e-prop이 생략하는 항(순환 Jacobian)이 작기 때문이라고 했습니다. 그 말을 직접 시험해 봅니다.

은닉층에 **순환 연결** $W_{rec}$ (hidden→hidden)를 넣어 전류를 $I[t] = x[t]\,W_{in}^\top + z[t{-}1]\,W_{rec}^\top$ 로 바꾸면, 은닉 활동이 자기 과거에 의존하게 됩니다.

- $W_{rec}$: 은닉→은닉 **순환 가중치**. 크기(scale)를 0으로 두면 앞의 피드포워드와 같습니다.
- $z[t{-}1]$: **직전 스텝**의 은닉 스파이크(그 활동이 이번 스텝 전류로 되먹임됩니다).

BPTT는 이 순환 경로의 gradient까지 전부 셈하지만, 우리가 쓴 **단순 e-prop의 자격 흔적은 입력 흔적만** 담고 순환 항을 무시합니다. 그래서 순환이 세질수록 둘의 방향이 갈릴 것으로 예상됩니다. $W_{rec}$ 크기를 키우며 $\cos(\text{e-prop}, \text{BPTT})$ 가 어떻게 변하는지 봅니다.

In [ ]:
# 실험: 은닉층에 순환 연결 W_rec를 넣으면 e-prop이 BPTT에서 얼마나 멀어지나
def forward_rec(x, W_in, W_out, W_rec):
    B = x.shape[1]; v = torch.zeros(B, H); z = torch.zeros(B, H)
    out = torch.zeros(B, C); eps = torch.zeros(B, n_in)
    psi_rec, eps_rec, z_rec = [], [], []
    for t in range(T):
        I = x[t] @ W_in.t() + z @ W_rec.t()            # 순환 입력: 직전 스텝 z가 되먹임
        v = alpha*v + I - z*v_th; z = spike(v); out = out + z @ W_out.t()
        with torch.no_grad():
            psi = gamma*torch.clamp(1 - torch.abs(v - v_th)/v_th, min=0); eps = alpha*eps + x[t]
            psi_rec.append(psi.clone()); eps_rec.append(eps.clone()); z_rec.append(z.detach().clone())
    return out/T, psi_rec, eps_rec, z_rec

x, y = gen_batch(128)
scales = [0.0, 0.1, 0.2, 0.35, 0.5]
coss = []
for sc in scales:
    torch.manual_seed(2)
    Wi = (torch.rand(H, n_in)*0.3).requires_grad_(True)
    Wo = torch.rand(C, H)*0.3 - 0.15
    Wr = torch.rand(H, H)*sc - sc/2                     # 순환 가중치 (sc=0이면 피드포워드)
    logits, psi_rec, eps_rec, z_rec = forward_rec(x, Wi, Wo, Wr)
    torch.nn.functional.cross_entropy(logits, y).backward()      # BPTT: 순환 경로까지 전부
    gWin_bptt = Wi.grad.clone()
    pi = torch.softmax(logits.detach(), 1); oh = torch.zeros(128, C); oh[torch.arange(128), y] = 1.0
    err = (pi - oh)/128; L = err @ Wo.detach()
    E = torch.zeros(128, H, n_in)
    for t in range(T): E += psi_rec[t].unsqueeze(2) * eps_rec[t].unsqueeze(1)
    gWin_ep = (L.unsqueeze(2) * E).sum(0)/T             # e-prop: 입력 자격 흔적만 (순환 무시)
    a, b = gWin_ep.flatten(), gWin_bptt.flatten()
    coss.append(float((a @ b)/(a.norm()*b.norm() + 1e-9)))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(scales, coss, marker="o", color="tab:purple")
ax.set_xlabel("순환 가중치 크기 (W_rec scale)"); ax.set_ylabel(r"$\cos$(e-prop, BPTT)")
ax.set_ylim(min(coss) - 0.02, 1.01)
ax.set_title("순환이 세질수록 e-prop이 BPTT에서 멀어진다 (근사의 한계)")
plt.tight_layout(); plt.show()
for sc, c in zip(scales, coss): print(f"  W_rec scale={sc}: cos = {c:.3f}")
print("=> 순환이 없으면 ~1.0(e-prop=BPTT), 커질수록 정렬이 떨어진다 = e-prop은 '근사'")

## 9. 정리 & 다음 단계

### 배운 것
- **surrogate gradient** $\psi$는 스파이크의 미분 불가를 우회 — **BPTT와 e-prop 둘 다** 사용.
- **e-prop**: $\dfrac{dE}{dW_{ji}}=\sum_t L_j^t e_{ji}^t$. 자격 흔적(지역·순방향) × 학습 신호(오차 사영). **시간 역방향·전체 히스토리 저장 불필요**(온라인·지역).
- **feedback alignment**: e-prop의 학습 신호가 되쓰던 readout 가중치를 **고정 랜덤 $B$**(갱신 안 함)로 바꿔도 학습됨 — $W^{out}$이 $B$에 정렬되며 **weight transport까지 제거**(Lillicrap et al., 2016; 오차를 각 층에 방송하는 변형은 DFA, Nøkland 2016).
- **DECOLLE**: 층마다 **고정 랜덤 국소 readout $Y^l$** 로 자기 손실 $\mathcal{L}^l$ 을 만들어, 출력 오차를 안 보고 **공간적으로도 지역**하게 학습(Kaiser et al., 2020). 실측: 은닉층이 국소 손실만으로 분류 가능한 특징을 학습(정확도 0.36→1.0).
- 실측: e-prop gradient가 **BPTT와 cosine ≈ 1** (얕은 net). e-prop만으로 학습 성공.
- 스칼라 보상에서 **뉴런별 학습 신호**로 세 번째 인자가 정교해졌고, **eligibility trace가 STDP·R-STDP·e-prop을 관통하는 한 줄기**임을 확인.
- **한 흐름**: 전역 gradient(BPTT)에서 출발해 **시간 지역화(e-prop) → weight transport 제거(feedback alignment) → 공간 지역화(DECOLLE)** 로, 같은 목표를 점점 더 지역적·생물학적으로 좁혀왔다.

### 다음 편(`06`) — Predictive Coding
지금까지는 보상/오차라는 명시적 신호로 배웠습니다. 다음 편은 신호의 출처를 바꿉니다: **위 층이 아래 층을 예측하고, 그 예측 오차로 지역 학습**하는 **predictive coding**. BPTT가 필요 없는 또 다른 지역 학습입니다.